In [ ]:
#| hide
using MIM_GA

In [ ]:
#| default_exp morphometrics

In [ ]:
#| export
using Rasters
import ArchGDAL

In [ ]:
#| export 
struct Bathymetry{R <: Raster} <: AbstractBathymetry
    raster::R 
end

In [ ]:
#| export
# Functor variant 1: Loads as a Bathymetry Depth Map
function (sm::SeafloorMap)(::Type{Bathymetry}; kwargs...)
    return Bathymetry(Raster(sm.path; name=:depth, kwargs...))    
end

In [ ]:
#| export
# Calculate resolution
function cellsize(bathy::Bathymetry; unit::AbstractString="meter")
    x_coords = lookup(bathy.raster, X)
    y_coords = lookup(bathy.raster, Y)

    clean_raster = replace_missing(bathy.raster, NaN)

    xres, yres = calc_resolution(Float64.(x_coords), Float64.(y_coords))
    cx, cy = cal_cellsize(xres, yres; unit=unit)
    
    scaled_bathy = clean_raster .* UNIT_TO_METERS[unit]

    return (cx, cy, Bathymetry(scaled_bathy)) 
    
end

cellsize (generic function with 1 method)

In [ ]:
#| export
# TPI calculations with cell size
function tpi(cellsize_x::Real, cellsize_y::Real, bathy::Bathymetry, radius::Real)
    clean_raster = replace_missing(bathy.raster, NaN)

    # 2. Extract a standard, pure in-memory Matrix{Float64} payload
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    kernel = circle_kernel(cellsize_x, cellsize_y, radius)

    tpi_matrix = kernel_mean_diff(raw_matrix, kernel)

    # 4. Rebuild the georeferenced output layer
    result_raster = rebuild(clean_raster; data = tpi_matrix, name = :tpi, missingval = NaN)
    
    return Bathymetry(result_raster) 
end

# TPI calculations with cell size
function tpi(bathy::Bathymetry, radius::Real; unit="meter")        
    #calculate for cell size
    cellsize_x, cellsize_y, scaled_bathy = cellsize(bathy, unit=unit)
    
    return tpi(cellsize_x, cellsize_y, scaled_bathy, radius)
end


Bathymetry{Raster{Union{Missing, Float32}, 2, Tuple{X{Projected{Float64, Rasters.StableRange{Float64}, DimensionalData.Dimensions.Lookups.ForwardOrdered, DimensionalData.Dimensions.Lookups.Regular{Float64}, DimensionalData.Dimensions.Lookups.Intervals{DimensionalData.Dimensions.Lookups.Start}, DimensionalData.Dimensions.Lookups.NoMetadata, WellKnownText{GeoFormatTypes.CRS}, Nothing, X{Colon}}}, Y{Projected{Float64, Rasters.StableRange{Float64}, DimensionalData.Dimensions.Lookups.ReverseOrdered, DimensionalData.Dimensions.Lookups.Regular{Float64}, DimensionalData.Dimensions.Lookups.Intervals{DimensionalData.Dimensions.Lookups.Start}, DimensionalData.Dimensions.Lookups.NoMetadata, WellKnownText{GeoFormatTypes.CRS}, Nothing, Y{Colon}}}}, Tuple{Band{DimensionalData.Dimensions.Lookups.Categorical{Int64, UnitRange{Int64}, DimensionalData.Dimensions.Lookups.ForwardOrdered, DimensionalData.Dimensions.Lookups.NoMetadata}}}, Matrix{Union{Missing, Float32}}, Symbol, DimensionalData.Dimensions.Loo